# Integration

In [ ]:
import sys
sys.path.insert(0, '../lib')

import os
import scvi
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import anndata as ad
import pynndescent
import numba
import seaborn as sns
import torch
import pytorch_lightning.loggers

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/b1196/envs/serniczek/lib/python3.10/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/projects/b1196/envs/serniczek/lib/python3.10/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (


In [ ]:
neptune_logger = pytorch_lightning.loggers.NeptuneLogger(
    project="nupulmonary/serniczek",
    log_model_checkpoints=False,
    name='data_v3_round_1_latent_10'
)

In [3]:
scvi.settings.seed = 1066

[rank: 0] Global seed set to 1066


In [ ]:
ds = sc.read_h5ad(common_data._sc_root / '02_hvg1000.h5ad')

In [5]:
ds.shape

(3017150, 1000)

In [6]:
scvi.model.SCVI.setup_anndata(ds, batch_key="individual")

/projects/b1196/envs/serniczek/lib/python3.10/abc.py:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.

  return _abc_instancecheck(cls, instance)
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [7]:
scvi.__version__

'1.0.4'

In [ ]:
model_kwargs = dict(
    n_latent=10,
    n_layers=2,
    dropout_rate=0.2,
    gene_likelihood="nb",
    use_observed_lib_size=False,
    encode_covariates=True,
    deeply_inject_covariates=False
)

In [ ]:
vae = scvi.model.SCVI(
    ds,
    **model_kwargs
)

In [11]:
trainer_kwargs = dict(
    accelerator='gpu',
    devices=1,
    max_epochs=400,
    check_val_every_n_epoch=5,
    early_stopping=True,
    gradient_clip_val=1.0,
    logger=neptune_logger
)

In [12]:
plan_kwargs = dict(
    optimizer='AdamW',
    lr=0.001,
)

In [13]:
torch.cuda.device_count()

2

In [14]:
log_params = dict(batch_size=2**15)
log_params.update(model_kwargs)
log_params.update(trainer_kwargs)
log_params.update(plan_kwargs)

In [15]:
neptune_logger.log_hyperparams(params=log_params)

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/pytorch_lightning/loggers/neptune.py:383: The following monitoring options are disabled by default in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', and 'capture_hardware_metrics'. To enable them, set each parameter to 'True' when initializing the run. The monitoring will continue until you call run.stop() or the kernel stops. Also note: Your source files can only be tracked if you pass the path(s) to the 'source_code' argument. For help, see the Neptune docs: https://docs.neptune.ai/logging/source_code/


https://app.neptune.ai/nupulmonary/serniczek/e/SER-13


/projects/b1196/envs/serniczek/lib/python3.10/site-packages/neptune/internal/utils/git.py:71: UserWarning: GitPython could not be initialized
  warnings.warn("GitPython could not be initialized")


In [ ]:
vae.train(
    batch_size=2**15,
    plan_kwargs=plan_kwargs,
    **trainer_kwargs,
)

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:168: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.10 /projects/b1196/envs/serniczek/lib/python3.10/si ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/projects/b1196/envs/serniczek/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:168: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.10 /projects/b1196/envs/serniczek/lib/python3.10/si ...
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precisio

Epoch 1/400:   0%|                                                                                                  | 0/400 [00:00<?, ?it/s]

/projects/b1196/envs/serniczek/lib/python3.10/abc.py:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.

  return _abc_instancecheck(cls, instance)


Epoch 16/400:   4%|█▏                             | 15/400 [04:03<1:42:53, 16.04s/it, v_num=R-13, train_loss_step=810, train_loss_epoch=819]

In [ ]:
vae.save("03_scvi_1000_10.model", override=False)